# Stage 4 — Phase 4 Zero-Shot: Molmo2-O-7B Only

Self-contained (no `git clone`/`src` import — just `git pull` + re-run the affected
cell for updates). Runs Molmo2 zero-shot over the real **20 frozen Gupta test sheets**
(tiled per checklist 2.3), stitches+dedups per-tile predictions back to sheet level
(checklist 3.3), and scores real detection precision/recall/F1 against ground truth
(checklist 3.1/3.2 — point-in-GT-box matching).

This is the real Phase 4 deliverable for Molmo2 specifically (deferred for all 3
candidates earlier; doing just Molmo2 now since it's cheap — ~10min per the 2.6
budget estimate — and its zero-shot format reliability was far better than the other
two). Goal: get a real, measured answer on whether Molmo2 is a good Stage 4 candidate,
rather than reasoning from format-compliance alone.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: User cancelled dfs_ephemeral authorization

## 2. Check GPU before installing anything

Run this first — if it errors or shows nothing, the runtime has no GPU (change
Runtime type before continuing, per the earlier CPU-runtime incident).

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

NVIDIA A100-SXM4-80GB, 81920 MiB


## 3. Install dependencies

Molmo2-O-7B requires `transformers==4.57.1` exactly (its own model card pins this).
**Deliberately NOT installing `torch` generically** — Colab's preinstalled CUDA build
gets silently replaced by a CPU-only PyPI wheel if you do (bit us twice already this
session).

In [3]:
!pip install -q transformers==4.57.1 accelerate

## ⚠️ Restart the runtime now (Runtime → Restart session), then continue from below

## 4. Config + shared imports

In [1]:
DRIVE_ROOT = "/content/drive/MyDrive/pid_project/data"
GUPTA_DIR = "gupta_pid"

import json, re, time
from pathlib import Path
import torch
from PIL import Image, ImageDraw
from transformers import (
    AutoModelForImageTextToText, AutoProcessor,
    MaxTimeCriteria, StoppingCriteriaList,
)

ROOT = Path(DRIVE_ROOT)
gupta_p = ROOT / GUPTA_DIR
print("Gupta:", gupta_p.exists())
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU — check Runtime type before continuing"

Gupta: True
torch: 2.11.0+cu128 | CUDA available: True


## 5. Tiling (checklist 2.3) + scoring/stitch harness (checklist 3.1-3.3) — inline

In [2]:
TILE_SIZE, OVERLAP = 1024, 205

def compute_tile_grid(img_w, img_h, tile_size=TILE_SIZE, overlap=OVERLAP):
    stride = tile_size - overlap
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1 = min(y0 + tile_size, img_h)
        x0 = 0
        while x0 < img_w:
            x1 = min(x0 + tile_size, img_w)
            tiles.append({"x0": x0, "y0": y0, "x1": x1, "y1": y1})
            x0 += stride
        y0 += stride
    return tiles

def yolo_line_to_xyxy(line, img_w, img_h):
    parts = line.split()
    cx, cy, w, h = (float(v) for v in parts[1:5])
    cx, cy, w, h = cx * img_w, cy * img_h, w * img_w, h * img_h
    return [cx - w/2, cy - h/2, cx + w/2, cy + h/2]

def load_gt_boxes(label_path, img_w, img_h):
    """Absolute sheet-coordinate GT boxes (class-agnostic — Gupta rule 5)."""
    if not label_path.exists():
        return []
    return [yolo_line_to_xyxy(l, img_w, img_h) for l in label_path.read_text().splitlines() if l.strip()]

# --- scoring.py (checklist 3.1/3.2) ---
def _bbox_center(bbox):
    x0, y0, x1, y1 = bbox
    return (x0 + x1) / 2, (y0 + y1) / 2

def point_in_box(point, box):
    x, y = point
    x0, y0, x1, y1 = box
    return x0 <= x <= x1 and y0 <= y <= y1

def prediction_point(pred):
    if pred.get("point") is not None:
        return tuple(pred["point"])
    return _bbox_center(pred["bbox"])

def match_predictions_to_gt(predictions, gt_boxes):
    order = sorted(range(len(predictions)),
                   key=lambda i: (predictions[i].get("confidence") is None, -(predictions[i].get("confidence") or 0)))
    gt_taken = [False] * len(gt_boxes)
    n_matched = 0
    for i in order:
        pt = prediction_point(predictions[i])
        for j, box in enumerate(gt_boxes):
            if not gt_taken[j] and point_in_box(pt, box):
                gt_taken[j] = True
                n_matched += 1
                break
    return n_matched, len(predictions), len(gt_boxes)

def precision_recall_f1(predictions, gt_boxes):
    n_matched, n_pred, n_gt = match_predictions_to_gt(predictions, gt_boxes)
    precision = n_matched / n_pred if n_pred else (1.0 if n_gt == 0 else 0.0)
    recall = n_matched / n_gt if n_gt else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "n_matched": n_matched, "n_pred": n_pred, "n_gt": n_gt}

# --- stitch.py (checklist 3.3) ---
DEDUP_DISTANCE_PX = 30

def remap_tile_to_sheet(pred, tile_origin):
    ox, oy = tile_origin
    out = dict(pred)
    x0, y0, x1, y1 = pred["bbox"]
    out["bbox"] = [x0 + ox, y0 + oy, x1 + ox, y1 + oy]
    if pred.get("point") is not None:
        px, py = pred["point"]
        out["point"] = [px + ox, py + oy]
    return out

def stitch_and_dedup(tile_predictions, tile_origins, dedup_distance_px=DEDUP_DISTANCE_PX):
    sheet_preds = []
    for preds, origin in zip(tile_predictions, tile_origins):
        for p in preds:
            sheet_preds.append(remap_tile_to_sheet(p, origin))
    clusters = []
    for pred in sheet_preds:
        pt = prediction_point(pred)
        etype = pred.get("entity_type")
        placed = False
        for cluster in clusters:
            if cluster["type"] != etype:
                continue
            cx, cy = prediction_point(cluster["members"][0])
            if ((pt[0]-cx)**2 + (pt[1]-cy)**2)**0.5 <= dedup_distance_px:
                cluster["members"].append(pred)
                placed = True
                break
        if not placed:
            clusters.append({"type": etype, "members": [pred]})
    deduped = []
    for cluster in clusters:
        members = cluster["members"]
        with_conf = [m for m in members if m.get("confidence") is not None]
        best = max(with_conf, key=lambda m: m["confidence"]) if with_conf else members[0]
        deduped.append(best)
    return deduped, len(sheet_preds)

## 6. Molmo2-O-7B: load, prompt, parse

In [3]:
MOLMO_MODEL_ID = "allenai/Molmo2-O-7B"

MOLMO_PROMPT = (
    "Point to every symbol (valve, instrument, flange, nozzle, safety device, or other "
    "P&ID symbol) visible in this image."
)

_POINTS_RE = re.compile(r'<(?:points|tracks).*? coords="([0-9\t:;, .]+)"/?>')

def load_molmo():
    t0 = time.time()
    processor = AutoProcessor.from_pretrained(MOLMO_MODEL_ID, trust_remote_code=True, dtype="auto")
    model = AutoModelForImageTextToText.from_pretrained(
        MOLMO_MODEL_ID, trust_remote_code=True, dtype="auto", device_map="cuda"
    )
    print(f"Loaded {MOLMO_MODEL_ID} in {time.time()-t0:.1f}s")
    print("VRAM used:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")
    return processor, model

def run_molmo(processor, model, image, prompt=MOLMO_PROMPT, max_time_s=60.0):
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}, {"type": "image", "image": image}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    ).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=2048, do_sample=False,
            stopping_criteria=StoppingCriteriaList([MaxTimeCriteria(max_time=max_time_s)]),
        )
    latency = time.time() - t0
    gen_tokens = out[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(gen_tokens, skip_special_tokens=True)[0]
    return text, latency

def parse_molmo(text, img_w, img_h):
    detections = []
    for m in _POINTS_RE.finditer(text):
        nums = [float(v) for v in re.split(r'[\t:;, ]+', m.group(1).strip()) if v]
        if len(nums) % 3 != 0:
            if len(nums) >= 2 and nums[0] == nums[1] and (len(nums)-1) % 3 == 0:
                nums = nums[1:]
            else:
                return None, f"coords not a multiple of 3: {nums}"
        for i in range(0, len(nums), 3):
            _frame, x_scaled, y_scaled = nums[i:i+3]
            x, y = x_scaled/1000*img_w, y_scaled/1000*img_h
            detections.append({"bbox": [x-20, y-20, x+20, y+20], "confidence": None, "entity_type": None, "point": [x, y]})
    if not detections and "<point" in text.lower():
        return None, "contains point-like tags but regex found no matches"
    return detections, None

## 7. Load model, run zero-shot over all 20 frozen test sheets, score

Per checklist 2.6: ~127 tiles total across 20 sheets, ~4.4s/tile measured earlier for
Molmo2 zero-shot → expect roughly 10 minutes.

In [5]:
processor, model = load_molmo()

test_ids = json.loads((ROOT / "test_ids.json").read_text())["test_ids"]
raw = gupta_p / "PID_Dataset" / "0__raw_data"

sheet_results = []
totals = {"n_matched": 0, "n_pred": 0, "n_gt": 0}
parse_failures = 0
total_tiles = 0

for sheet_id in test_ids:
    img_path = next((raw / "sheets" / "test").glob(f"{sheet_id}.*"))
    label_path = raw / "labels" / "test" / f"{sheet_id}.txt"
    sheet_img = Image.open(img_path).convert("RGB")
    W, H = sheet_img.size
    gt_boxes = load_gt_boxes(label_path, W, H)

    tiles = compute_tile_grid(W, H)
    tile_preds, tile_origins = [], []
    for t in tiles:
        total_tiles += 1
        crop = sheet_img.crop((t["x0"], t["y0"], t["x1"], t["y1"]))
        raw_text, _latency = run_molmo(processor, model, crop)
        dets, err = parse_molmo(raw_text, t["x1"]-t["x0"], t["y1"]-t["y0"])
        if err:
            parse_failures += 1
            dets = []
        tile_preds.append(dets)
        tile_origins.append((t["x0"], t["y0"]))

    sheet_preds, n_raw = stitch_and_dedup(tile_preds, tile_origins)
    result = precision_recall_f1(sheet_preds, gt_boxes)
    sheet_results.append({"sheet_id": sheet_id, **result})
    for k in totals:
        totals[k] += result[k]
    print(f"{sheet_id:10s} tiles={len(tiles):2d} gt={result['n_gt']:3d} pred={result['n_pred']:3d} "
          f"matched={result['n_matched']:3d} precision={result['precision']:.2f} recall={result['recall']:.2f} f1={result['f1']:.2f}")

print(f"\n--- Overall (micro-averaged across all {len(test_ids)} test sheets, {total_tiles} tiles) ---")
overall_precision = totals["n_matched"] / totals["n_pred"] if totals["n_pred"] else 0.0
overall_recall = totals["n_matched"] / totals["n_gt"] if totals["n_gt"] else 0.0
overall_f1 = 2*overall_precision*overall_recall/(overall_precision+overall_recall) if (overall_precision+overall_recall) else 0.0
print(f"Precision: {overall_precision:.3f} | Recall: {overall_recall:.3f} | F1: {overall_f1:.3f}")
print(f"Parse failures: {parse_failures}/{total_tiles} tiles ({100*parse_failures/total_tiles:.1f}%)")
print(f"\nProvisional pass bar (Stage4_Checklist_Status.md 3.6): F1 >= 0.70")
print("✓ MEETS bar" if overall_f1 >= 0.70 else "✗ below bar")

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 172.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 43.19 MiB is free. Process 10627 has 32.51 GiB memory in use. Process 37358 has 16.78 GiB memory in use. Including non-PyTorch memory, this process has 29.89 GiB memory in use. Of the allocated memory 29.43 GiB is allocated by PyTorch, and 57.94 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)